<a href="https://colab.research.google.com/github/USHODAYAKALYANI/LSTM-Prediction/blob/main/TF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, MinMaxScaler

import tensorflow as tf

In [ ]:
train_df = pd.read_excel('/content/train.xlsx')
val_df = pd.read_excel('/content/val.xlsx')
test_df = pd.read_excel('/content/test.xlsx')

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(302374, 14)
(81605, 14)
(81606, 14)


In [ ]:
features = ['Speed','Ax','Ay','Az','Gx','Gy','Gz']

In [ ]:
X_train = train_df[features]
y_train = train_df['Label']

X_val = val_df[features]
y_val = val_df['Label']

X_test = test_df[features]
y_test = test_df['Label']



In [ ]:
le = LabelEncoder()

y_train = le.fit_transform(y_train)

y_val = le.transform(y_val)

y_test = le.transform(y_test)

print(le.classes_)

['BUMP' 'LEFT' 'RIGHT' 'STOP' 'STRAIGHT']


In [ ]:
scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)

X_val = scaler.transform(X_val)

X_test = scaler.transform(X_test)

window = 20

In [ ]:
from collections import Counter

def create_sequences(X, y, window):

    X_seq = []
    y_seq = []

    for i in range(len(X)-window):

        X_seq.append(X[i:i+window])

        counts = Counter(y[i:i+window])

        majority_label = counts.most_common(1)[0][0]

        y_seq.append(majority_label)

    return np.array(X_seq), np.array(y_seq)

In [ ]:
window = 20

X_train_seq, y_train_seq = create_sequences(X_train, y_train, window)

X_val_seq, y_val_seq = create_sequences(X_val, y_val, window)

X_test_seq, y_test_seq = create_sequences(X_test, y_test, window)

In [ ]:
print(X_train_seq.shape)
print(y_train_seq.shape)

print(X_val_seq.shape)
print(y_val_seq.shape)

print(X_test_seq.shape)
print(y_test_seq.shape)

(302354, 20, 7)
(302354,)
(81585, 20, 7)
(81585,)
(81586, 20, 7)
(81586,)


In [ ]:
import tensorflow as tf

inputs = tf.keras.Input(shape=(20, 7))

x = tf.keras.layers.MultiHeadAttention(
    num_heads=4,
    key_dim=16
)(inputs, inputs)

x = tf.keras.layers.LayerNormalization(
    epsilon=1e-6
)(x + inputs)

ffn = tf.keras.layers.Dense(
    64,
    activation='relu'
)(x)

ffn = tf.keras.layers.Dense(7)(ffn)

x = tf.keras.layers.LayerNormalization(
    epsilon=1e-6
)(x + ffn)

x = tf.keras.layers.GlobalAveragePooling1D()(x)

x = tf.keras.layers.Dense(
    64,
    activation='relu'
)(x)

x = tf.keras.layers.Dropout(0.2)(x)

outputs = tf.keras.layers.Dense(
    len(le.classes_),
    activation='softmax'
)(x)

model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs
)

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 20, 7)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 20, 7)     │      1,991 │ input_layer[0][0… │
│ (MultiHeadAttentio… │                   │            │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 20, 7)     │          0 │ multi_head_atten… │
│                     │                   │            │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 20, 7)     │         14 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 20, 64)    │        512 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 20, 7)     │        455 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 20, 7)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 20, 7)     │         14 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 7)         │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │        512 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 5)         │        325 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,823 (14.93 KB)

 Trainable params: 3,823 (14.93 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=10,
    batch_size=64
)

Epoch 1/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 31s 5ms/step - accuracy: 0.8291 - loss: 0.5147 - val_accuracy: 0.8231 - val_loss: 0.5814
Epoch 2/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.8419 - loss: 0.4576 - val_accuracy: 0.8202 - val_loss: 0.5577
Epoch 3/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.8462 - loss: 0.4438 - val_accuracy: 0.8236 - val_loss: 0.5590
Epoch 4/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - accuracy: 0.8493 - loss: 0.4330 - val_accuracy: 0.7911 - val_loss: 0.6298
Epoch 5/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.8509 - loss: 0.4257 - val_accuracy: 0.7615 - val_loss: 0.6639
Epoch 6/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - accuracy: 0.8525 - loss: 0.4200 - val_accuracy: 0.8128 - val_loss: 0.6085
Epoch 7/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - accuracy: 0.8535 - loss: 0.4163 - val_accuracy: 0.7798 - val_loss: 0.6478
Epoch 8/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - accuracy: 0.8547 - loss: 0

In [ ]:
X_trainval = np.concatenate(
    [X_train_seq, X_val_seq],
    axis=0
)

y_trainval = np.concatenate(
    [y_train_seq, y_val_seq],
    axis=0
)


In [ ]:
print(X_trainval.shape)
print(y_trainval.shape)

(383939, 20, 7)
(383939,)


In [ ]:
model.fit(
    X_trainval,
    y_trainval,
    epochs=10,
    batch_size=64
)

Epoch 1/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 24s 4ms/step - accuracy: 0.8503 - loss: 0.4353
Epoch 2/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - accuracy: 0.8522 - loss: 0.4312
Epoch 3/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 22s 4ms/step - accuracy: 0.8531 - loss: 0.4288
Epoch 4/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 39s 3ms/step - accuracy: 0.8535 - loss: 0.4265
Epoch 5/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 21s 4ms/step - accuracy: 0.8547 - loss: 0.4241
Epoch 6/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 21s 4ms/step - accuracy: 0.8552 - loss: 0.4217
Epoch 7/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - accuracy: 0.8559 - loss: 0.4190
Epoch 8/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 21s 4ms/step - accuracy: 0.8574 - loss: 0.4162
Epoch 9/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - accuracy: 0.8574 - loss: 0.4148
Epoch 10/10
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - accuracy: 0.8592 - loss: 0.4116


In [ ]:
test_loss, test_acc = model.evaluate(
    X_test_seq,
    y_test_seq
)

print("Test Accuracy:", test_acc)

2550/2550 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.6390 - loss: 1.3649
Test Accuracy: 0.6389821767807007


TST

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

class PositionalEncoding(layers.Layer):

    def __init__(self, sequence_length, d_model):
        super().__init__()

        self.pos_embedding = layers.Embedding(
            input_dim=sequence_length,
            output_dim=d_model
        )

        self.sequence_length = sequence_length

    def call(self, x):

        positions = tf.range(
            start=0,
            limit=self.sequence_length,
            delta=1
        )

        pos_embeddings = self.pos_embedding(positions)

        return x + pos_embeddings

In [ ]:
def transformer_encoder(
        inputs,
        head_size,
        num_heads,
        ff_dim,
        dropout=0.2):

    x = layers.MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout
    )(inputs, inputs)

    x = layers.Dropout(dropout)(x)

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(inputs + x)

    ffn = layers.Dense(
        ff_dim,
        activation="relu"
    )(x)

    ffn = layers.Dense(
        inputs.shape[-1]
    )(ffn)

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x + ffn)

    return x

In [ ]:
inputs = tf.keras.Input(shape=(20,7))

x = layers.Dense(64)(inputs)

x = PositionalEncoding(
    sequence_length=20,
    d_model=64
)(x)

for _ in range(2):

    x = transformer_encoder(
        x,
        head_size=16,
        num_heads=4,
        ff_dim=128,
        dropout=0.2
    )

x = layers.GlobalAveragePooling1D()(x)

x = layers.Dense(
    64,
    activation='relu'
)(x)

x = layers.Dropout(0.2)(x)

outputs = layers.Dense(
    len(le.classes_),
    activation='softmax'
)(x)

tst_model = tf.keras.Model(
    inputs,
    outputs
)

In [ ]:
tst_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
tst_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 20, 7)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 20, 64)    │        512 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encoding │ (None, 20, 64)    │      1,280 │ dense_4[0][0]     │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 20, 64)    │     16,640 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 20, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 20, 64)    │          0 │ positional_encod… │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 20, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 20, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 20, 64)    │      8,256 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 20, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 20, 64)    │        128 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 20, 64)    │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 20, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 20, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 20, 64)    │        128 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 20, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 20, 64)    │      8,256 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 20, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 20, 64)    │        128 │ add_5[0][0]       │
│ (LayerNormalizatio… │                   │            │                 

 Total params: 73,221 (286.02 KB)

 Trainable params: 73,221 (286.02 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
tst_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = tst_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 46s 7ms/step - accuracy: 0.8336 - loss: 0.4962 - val_accuracy: 0.8182 - val_loss: 0.6642
Epoch 2/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 25s 5ms/step - accuracy: 0.8482 - loss: 0.4346 - val_accuracy: 0.8302 - val_loss: 0.5675
Epoch 3/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 26s 6ms/step - accuracy: 0.8552 - loss: 0.4099 - val_accuracy: 0.8112 - val_loss: 0.5901
Epoch 4/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 26s 5ms/step - accuracy: 0.8605 - loss: 0.3894 - val_accuracy: 0.8151 - val_loss: 0.5849
Epoch 5/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 26s 6ms/step - accuracy: 0.8656 - loss: 0.3722 - val_accuracy: 0.7976 - val_loss: 0.6278
Epoch 6/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 26s 5ms/step - accuracy: 0.8719 - loss: 0.3518 - val_accuracy: 0.8277 - val_loss: 0.5947
Epoch 7/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 26s 6ms/step - accuracy: 0.8787 - loss: 0.3324 - val_accuracy: 0.7997 - val_loss: 0.7037
Epoch 8/10
4725/4725 ━━━━━━━━━━━━━━━━━━━━ 40s 5ms/step - accuracy: 0.8850 - loss: 0

In [ ]:
X_trainval = np.concatenate(
    [X_train_seq, X_val_seq],
    axis=0
)

y_trainval = np.concatenate(
    [y_train_seq, y_val_seq],
    axis=0
)

In [ ]:
test_loss, test_acc = tst_model.evaluate(
    X_test_seq,
    y_test_seq
)

print("Test Accuracy:", test_acc)

2550/2550 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.6423 - loss: 1.8555
Test Accuracy: 0.6423161029815674
